In [45]:
import tensorflow as tf  # TensorFlowを読み込む
from pathlib import Path
IMG_SIZE = (160, 160)  # MobileNetV2に入力する画像サイズ（高さ・幅）
BATCH_SIZE = 16  # 画像を16枚ずつのバッチにまとめて処理する


In [46]:
# 学習用画像を読み込む（300枚のうち80％の240枚）
train_ds = tf.keras.utils.image_dataset_from_directory( 
    "dog_cat_photos/train",
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    class_names=["cat", "dog"]
)

Found 300 files belonging to 2 classes.


Using 240 files for training.


In [47]:
# 検証用画像を読み込む（300枚のうち20％の60枚）
val_ds = tf.keras.utils.image_dataset_from_directory(
    "dog_cat_photos/train",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    class_names=["cat", "dog"]
)

Found 300 files belonging to 2 classes.
Using 60 files for validation.


In [48]:
#テスト用画像100枚読み込み
test_ds = tf.keras.utils.image_dataset_from_directory(
    "dog_cat_photos/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    class_names=["cat", "dog"],
    shuffle=False
)

Found 100 files belonging to 2 classes.


In [49]:
data_augmentation = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal")]) #左右反転
data_augmentation.add(tf.keras.layers.RandomRotation(0.05)) #軽い回転

In [50]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

In [51]:
base_model = tf.keras.applications.MobileNetV2( 
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

In [52]:
inputs = tf.keras.Input(shape=IMG_SIZE + (3,)) 
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x) #分類しやすい形にまとめる

In [53]:
#犬猫判定
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
model = tf.keras.Model(inputs=inputs, outputs=outputs)


In [54]:
# 学習方法と評価方法の設定
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
#学習させる
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.8417 - loss: 0.4219

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.9800 - loss: 0.0528
